# 04 -- Visualization & Findings
**Project:** Airbnb Market Analysis -- New York City  
**Layer:** Gold (reporting)  
**Author:** B. Sarang  

Generates dashboard-ready outputs from Gold tables. Covers price distributions, neighbourhood trends, room type correlations, and host performance. Use Databricks AI/BI Dashboards to visualize outputs from each query cell.

## 1. Configuration

In [0]:
SILVER_TABLE = "workspace.default.silver_airbnb_listings"
GOLD_NEIGHBOURHOOD = "workspace.default.gold_neighbourhood_performance"
GOLD_ROOM_TYPE = "workspace.default.gold_room_type_summary"
GOLD_HOST = "workspace.default.gold_host_performance"

## 2. Price distribution by room type (bar chart)

In [0]:
spark.sql(f"""
    SELECT
        room_type,
        ROUND(AVG(price), 2)    AS avg_price,
        ROUND(MIN(price), 2)    AS min_price,
        ROUND(MAX(price), 2)    AS max_price,
        ROUND(STDDEV(price), 2) AS stddev_price,
        COUNT(*)                AS listing_count
    FROM {SILVER_TABLE}
    GROUP BY room_type
    ORDER BY avg_price DESC
""").display()

## 3. Top 15 neighbourhoods by average rating (horizontal bar)

In [0]:
spark.sql(f"""
    SELECT
        neighbourhood,
        borough,
        avg_rating,
        avg_price,
        total_listings,
        avg_est_monthly_revenue
    FROM {GOLD_NEIGHBOURHOOD}
    WHERE total_listings >= 10
    ORDER BY avg_rating DESC
    LIMIT 15
""").display()

## 4. Borough-level heatmap data -- avg price vs avg rating

In [0]:
spark.sql(f"""
    SELECT
        borough,
        room_type,
        ROUND(AVG(price), 2)                AS avg_price,
        ROUND(AVG(review_scores_rating), 2) AS avg_rating,
        ROUND(AVG(availability_365), 0)     AS avg_availability,
        COUNT(*)                            AS listing_count
    FROM {SILVER_TABLE}
    WHERE borough IS NOT NULL
    GROUP BY borough, room_type
    ORDER BY borough, avg_price DESC
""").display()

## 5. Demand trend -- listings by availability bucket (histogram data)

In [0]:
spark.sql(f"""
    SELECT
        CASE
            WHEN availability_365 BETWEEN 0 AND 60   THEN '0-60 days (high demand)'
            WHEN availability_365 BETWEEN 61 AND 180 THEN '61-180 days (moderate)'
            WHEN availability_365 BETWEEN 181 AND 300 THEN '181-300 days (low demand)'
            ELSE '300+ days (very low demand)'
        END AS availability_bucket,
        room_type,
        COUNT(*)                            AS listing_count,
        ROUND(AVG(price), 2)                AS avg_price,
        ROUND(AVG(review_scores_rating), 2) AS avg_rating
    FROM {SILVER_TABLE}
    GROUP BY availability_bucket, room_type
    ORDER BY availability_bucket, room_type
""").display()

## 6. Price vs rating correlation by borough (scatter data)

In [0]:
spark.sql(f"""
    SELECT
        borough,
        ROUND(CORR(price, review_scores_rating), 4) AS price_rating_corr,
        ROUND(CORR(price, reviews_per_month), 4)    AS price_reviews_corr,
        ROUND(CORR(availability_365, price), 4)     AS availability_price_corr,
        COUNT(*)                                    AS sample_size
    FROM {SILVER_TABLE}
    WHERE review_scores_rating IS NOT NULL
    GROUP BY borough
    ORDER BY borough
""").display()

## 7. Superhost impact on key metrics

In [0]:
spark.sql(f"""
    SELECT
        host_is_superhost,
        COUNT(*)                                    AS listing_count,
        ROUND(AVG(price), 2)                        AS avg_price,
        ROUND(AVG(review_scores_rating), 2)         AS avg_rating,
        ROUND(AVG(reviews_per_month), 2)            AS avg_reviews_per_month,
        ROUND(AVG(availability_365), 0)             AS avg_availability
    FROM {SILVER_TABLE}
    WHERE host_is_superhost IS NOT NULL
    GROUP BY host_is_superhost
""").display()

## 8. Geographic distribution -- listings by borough (map data)

In [0]:
spark.sql(f"""
    SELECT
        id,
        name,
        borough,
        neighbourhood,
        latitude,
        longitude,
        room_type,
        price,
        review_scores_rating
    FROM {SILVER_TABLE}
    WHERE latitude IS NOT NULL
      AND longitude IS NOT NULL
    LIMIT 5000
""").display()

## 9. Key findings summary

| Metric | Finding |
|---|---|
| Highest rated borough | Bronx (Woodlawn 4.94, Morris Park 4.92) |
| Most affordable room type | Shared room ($164 avg) -- Private room best value at $192 |
| Highest demand neighbourhood | Woodlawn, Bronx (4.94 rating, $348 est. monthly revenue) |
| Superhost avg rating vs non-superhost | Superhosts: 4.86 vs non-superhosts: 4.65 -- 75% more reviews/month (1.46 vs 0.84) |
| Strongest price-rating correlation borough | Staten Island (0.135) -- price and rating most positively correlated |